# 15 — A typed HTTP boundary
FastAPI accepts a validated request and calls the same application entry point
as the CLI. A public request never supplies its own approval.



## Problem

A CLI alone cannot serve another application safely.


## Concept

FastAPI validates requests and delegates to the existing app boundary.

HTTP validates input, requires a workshop token, and calls the same app function as the CLI. A separate instructor token protects mock approval.


## Abstraction

HTTP request → typed schema → run_assistant → Flow result. This is the smallest new layer introduced by this lesson; the earlier layers remain in place.


## Architecture

```text
caller → API token + Pydantic → run_assistant → Flow → PostgreSQL; instructor token → atomic mock approval
```

Follow the arrows while executing the cells. A tool result or validation failure is an observation, not an instruction that the model may bypass.


## Step 1 — Create a TestClient and confirm health and authentication boundaries

Create a TestClient and confirm health and authentication boundaries.


In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from fastapi.testclient import TestClient
from workshop.api import api

client = TestClient(api)
print("Health:", client.get("/health").json())
print("Unauthenticated request status:", client.post("/decisions", json={
    "question": "Should Northstar AI pilot Meridian AI?", "customer_id": "customer-a"
}).status_code)



## Run with a real model only when the instructor has configured it
Set WORKSHOP_API_TOKEN and Ollama first. This call may take time.



## Step 2 — Opt in to a live CrewAI run through the API

Opt in to a live CrewAI run through the API.


In [ ]:
if os.getenv("WORKSHOP_RUN_LIVE_DEMOS") == "true":
    token = os.environ["WORKSHOP_API_TOKEN"]
    response = client.post("/decisions", headers={"X-Workshop-API-Token": token}, json={
        "question": "Should Northstar AI pilot Meridian AI?", "customer_id": "customer-a"
    })
    print(response.status_code, response.json())
else:
    print("Set WORKSHOP_RUN_LIVE_DEMOS=true to call the local model through the API.")


## Execution flow

HTTP request → typed schema → run_assistant → Flow result. Read the printed result together with the task, selected tool, state, or trace. If the result differs, locate the first boundary where the observed value changed.


## Failure mode to know about

Accepting an approved field from a caller would let the request grant its own permission. The API accepts no such field and requires separate tokens.


## Production implication

Use real per-user authentication before exposing beyond localhost. Never accept approval from request JSON.


## Your turn

Send a request without the API token, then with it on a live setup.

**Checkpoint:** Unauthenticated access is denied; the live path has no client-supplied approval.
